# 02 · Train FHE-friendly CNN
Train the shallow CNN on FER2013 with Square ($x^2$) activations and strided convolution.
**Configuration**:
- 16 channels: Balanced feature extraction with fast inference
- FC1: 128 nodes: Sufficient representation capacity  
- Learning Rate Scheduler: Better convergence
- 50 epochs: Optimal training duration

Architecture: Conv2d(16ch, s3) -> Square -> Flatten -> FC(128) -> Square -> FC(7).
Multiplicative Depth: 3 (FHE-compatible), Inference time: ~18-20s

### 블록 1 · 라이브러리/모델 불러오기
학습에 필요한 PyTorch, NumPy, tqdm, 그리고 FHE 전용 CNN 모듈을 임포트합니다.


In [1]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f'Python path prepared with project root: {PROJECT_ROOT}')


Python path prepared with project root: /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion


In [2]:
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from tqdm.notebook import tqdm

from models.fhe_cnn import FHEEmotionCNN, extract_fhe_parameters

### 블록 2 · 경로 및 하이퍼파라미터 정의
데이터 위치, 저장 경로, 배치 크기와 에폭 수 등을 설정합니다.


In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_OUT = PROJECT_ROOT / 'models' / 'fhe_cnn_fer2013_enhanced.pt'
NORM_STATS_PATH = PROJECT_ROOT / 'models' / 'normalization_stats.json'
BATCH_SIZE = 64
EPOCHS = 50  # 최적 학습 기간
LR = 1e-3
WEIGHT_DECAY = 1e-5  # L2 정규화 추가
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print(f'Training config: {EPOCHS} epochs, LR={LR}, WD={WEIGHT_DECAY}')

Device: cpu
Training config: 70 epochs, LR=0.001, WD=1e-05


### 블록 3 · 전처리된 텐서 로딩
데이터 준비 노트북에서 저장한 이미지·레이블·클래스 가중치 텐서를 불러옵니다.


In [9]:
def load_tensor(name: str) -> torch.Tensor:
    path = DATA_DIR / f'{name}.pt'
    tensor = torch.load(path)
    print(f'Loaded {name} -> {tensor.shape}')
    return tensor

train_images = load_tensor('train_images')
val_images = load_tensor('val_images')
test_images = load_tensor('test_images')
train_labels = load_tensor('train_labels')
val_labels = load_tensor('val_labels')
test_labels = load_tensor('test_labels')
class_weights = load_tensor('class_weights')


Loaded train_images -> torch.Size([28709, 1, 48, 48])
Loaded val_images -> torch.Size([3589, 1, 48, 48])
Loaded test_images -> torch.Size([3589, 1, 48, 48])
Loaded train_labels -> torch.Size([28709])
Loaded val_labels -> torch.Size([3589])
Loaded test_labels -> torch.Size([3589])
Loaded class_weights -> torch.Size([7])


### 블록 4 · 정규화 통계 계산
학습 세트의 평균과 표준편차를 구해 JSON으로 저장하고 이후 노멀라이즈에 사용합니다.


In [10]:
train_mean = train_images.mean().item()
train_std = train_images.std().item()
print(f'Train mean: {train_mean:.4f}, std: {train_std:.4f}')
stats = {'mean': train_mean, 'std': train_std}
with open(NORM_STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
print('Saved normalization stats ->', NORM_STATS_PATH)


Train mean: 0.5072, std: 0.2550
Saved normalization stats -> /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/normalization_stats.json


### 블록 5 · 변환 및 데이터셋 구성
데이터 증강(Flip, Crop, Rotation) 파이프라인과 PyTorch Dataset/DataLoader를 정의합니다.


In [ ]:
base_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[train_mean], std=[train_std]),
])

# 기본 데이터 증강: 간단한 변형으로 빠른 학습과 안정적인 수렴
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),  # 50% 좌우 반전
    T.RandomAffine(
        degrees=10,  # ±10도 회전
        scale=(0.9, 1.0),  # 90-100% 크기 조정
    ),
    T.RandomResizedCrop(size=48, scale=(0.9, 1.0)),  # 기본 크롭 범위
    base_transform,
])

eval_transform = T.Compose([
    T.ToPILImage(),
    base_transform,
])

class AugmentedFERDataset(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        lbl = self.labels[idx]
        array = img.squeeze(0).numpy().astype(np.float32)
        if self.transform:
            img_tensor = self.transform(array)
        else:
            img_tensor = torch.tensor(array)[None, :, :]
            img_tensor = T.Normalize(mean=[train_mean], std=[train_std])(img_tensor)
        return img_tensor, lbl

train_dataset = AugmentedFERDataset(train_images, train_labels, transform=train_transform)
val_dataset = AugmentedFERDataset(val_images, val_labels, transform=eval_transform)
test_dataset = AugmentedFERDataset(test_images, test_labels, transform=eval_transform)
NUM_WORKERS = 0  # 노트북 환경에서는 multi-processing pickle 이슈 방지를 위해 0으로 둔다.
PIN_MEMORY = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train samples: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

Train samples: 28709, Val: 3589, Test: 3589


### 블록 6 · 모델 및 최적화 기법 설정
`FHEEmotionCNN`, 가중치가 적용된 CrossEntropyLoss, Adam 옵티마이저를 초기화합니다.


In [12]:
model = FHEEmotionCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Weight decay 추가로 과적합 방지
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Learning Rate Scheduler: Validation accuracy 정체 시 LR 감소
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',  # val_acc 최대화 목표
    factor=0.5,  # LR을 절반으로 감소
    patience=7,  # 7 epochs 동안 개선 없으면 감소
    verbose=True,
    min_lr=1e-6
)

best_val_acc = 0.0
history = []
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Model parameters: 1,207,479


In [13]:
# Verify model architecture and output shape
print(model)
dummy_input = torch.randn(1, 1, 48, 48).to(device)
with torch.no_grad():
    output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == (1, 7), f"Expected output shape (1, 7), got {output.shape}"


FHEEmotionCNN(
  (conv1): Conv2d(1, 24, kernel_size=(7, 7), stride=(3, 3))
  (act1): Square()
  (fc1): Linear(in_features=4704, out_features=256, bias=True)
  (act2): Square()
  (fc2): Linear(in_features=256, out_features=7, bias=True)
)
Input shape: torch.Size([1, 1, 48, 48])
Output shape: torch.Size([1, 7])


### 블록 7 · 학습 루프
에폭별로 학습/검증 손실·정확도를 계산하며 최적 모델을 저장합니다.


In [14]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch} / {EPOCHS}'):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        total += images.size(0)
    train_loss /= total
    train_acc = train_correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    # LR Scheduler step (validation accuracy 기반)
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']
    
    history.append({
        'epoch': epoch, 
        'train_loss': train_loss, 
        'train_acc': train_acc, 
        'val_loss': val_loss, 
        'val_acc': val_acc,
        'lr': current_lr
    })
    print(f'Epoch {epoch}: train_loss={train_loss:.4f} train_acc={train_acc:.3f} | val_loss={val_loss:.4f} val_acc={val_acc:.3f} | LR={current_lr:.6f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save best model
        torch.save(model.state_dict(), MODEL_OUT)
        print(f'✓ Saved new best model (val_acc={val_acc:.3f}) -> {MODEL_OUT.name}')

print(f'\n🎉 Training complete! Best validation accuracy: {best_val_acc:.3f}')

Epoch 1 / 70:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 1: train_loss=2.0302 train_acc=0.195 | val_loss=1.8839 val_acc=0.196 | LR=0.001000
✓ Saved new best model (val_acc=0.196) -> fhe_cnn_fer2013_enhanced.pt


Epoch 2 / 70:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 2: train_loss=1.8910 train_acc=0.251 | val_loss=1.8291 val_acc=0.320 | LR=0.001000
✓ Saved new best model (val_acc=0.320) -> fhe_cnn_fer2013_enhanced.pt


Epoch 3 / 70:   0%|          | 0/449 [00:00<?, ?it/s]

...

Epoch 35: train_loss=1.6127 train_acc=0.397 | val_loss=1.6344 val_acc=0.429 | LR=0.000500


Epoch 36 / 70:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 36: train_loss=1.6055 train_acc=0.399 | val_loss=1.5948 val_acc=0.446 | LR=0.000500


Epoch 37 / 70:   0%|          | 0/449 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 블록 8 · 테스트 평가
보존한 최적 가중치로 테스트 세트 정확도를 측정하고 히스토리를 출력합니다.


In [12]:
def evaluate(loader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
    return correct / total

test_acc = evaluate(test_loader)
print(f'Test accuracy: {test_acc:.3f}')
print('History:', history)


Test accuracy: 0.499
History: [{'epoch': 1, 'train_loss': 2.3138780273062927, 'train_acc': 0.26458601832178064, 'val_loss': 1.7625119276915662, 'val_acc': 0.342713847868487}, {'epoch': 2, 'train_loss': 1.7666249770539162, 'train_acc': 0.3456059075551221, 'val_loss': 1.6992766128738268, 'val_acc': 0.379771524101421}, {'epoch': 3, 'train_loss': 1.6870972616279123, 'train_acc': 0.37587516109930685, 'val_loss': 1.6971891090444085, 'val_acc': 0.40484814711618833}, {'epoch': 4, 'train_loss': 1.641814123666948, 'train_acc': 0.39308230868368804, 'val_loss': 1.6742974715200962, 'val_acc': 0.42323767066035106}, {'epoch': 5, 'train_loss': 1.6079296784580486, 'train_acc': 0.4000139329130238, 'val_loss': 1.5997918052871303, 'val_acc': 0.42602396210643634}, {'epoch': 6, 'train_loss': 1.5786616423938162, 'train_acc': 0.408443345292417, 'val_loss': 1.587577749457855, 'val_acc': 0.42518807467261077}, {'epoch': 7, 'train_loss': 1.543566031679223, 'train_acc': 0.41990316625448465, 'val_loss': 1.575119166

In [13]:
# Verify FHE parameter extraction
print("Extracting FHE parameters...")
params = extract_fhe_parameters(model)
print("Keys:", params.keys())
print("Conv layers:", len(params['conv']))
print("Linear layers:", len(params['linear']))
for i, layer in enumerate(params['conv']):
    print(f"Conv[{i}] weight shape: {layer['weight'].shape}, bias shape: {layer['bias'].shape}")
for i, layer in enumerate(params['linear']):
    print(f"Linear[{i}] weight shape: {layer['weight'].shape}, bias shape: {layer['bias'].shape}")


Extracting FHE parameters...
Keys: dict_keys(['conv', 'linear'])
Conv layers: 1
Linear layers: 2
Conv[0] weight shape: torch.Size([24, 1, 7, 7]), bias shape: torch.Size([24])
Linear[0] weight shape: torch.Size([128, 10584]), bias shape: torch.Size([128])
Linear[1] weight shape: torch.Size([7, 128]), bias shape: torch.Size([7])
